# Phase 6, track S (Syaz): training-recipe runs S1 + S2 (T4)

**What this notebook does, in plain words:** it trains the same filament detector Kimee used at 1024 px
(Mask R-CNN R50, run `r50_allann` = **exp006**, val PQ 0.3979) again, changing **one thing per run**, and saves its
predictions so we can score them on the laptop.

| Run | Name | What changes | Why | Time on T4 |
|---|---|---|---|---|
| 0 | `syaz_probe` | 20 training steps, then deleted | checks that everything is installed | ~10 min incl. install |
| **S1** | `syaz_r50_seed2` | nothing, only the random seed (7 instead of 42) | shows how much the score moves **just by re-training**. Any later "gain" smaller than this is luck | ~1.3 h |
| **S2** | `syaz_r50_3x` | trains 3× longer (18,000 steps instead of 6,000) | 6,000 steps is short for this model; longer training is the cheapest possible gain | ~4 h |

S3 (rotations + scale), S4 (copy-paste) and S5 (pseudo-labels) come later: Claude adds their code first, then a new
cell is added here.

**Where things go:** everything is written inside the team Drive folder **`solar_filament/Phase 6/Syaz/`** only
(checkpoints in `Phase 6/Syaz/output/`, prediction files next to it). Kimee's files are never touched.

**How to run:** Runtime → Change runtime type → **T4 GPU**. Run sections 1–4 and Step 0, then S1. S2 can run in the
same session or a later one. Keep the tab open.

**If Colab disconnects:** run sections 1–4 again, then re-run the cell of the interrupted run: training continues
from its last checkpoint (saved every 1,000 steps). A run that already finished is skipped automatically.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Google Drive (team folder)
Needs the shortcut to Kimee's `solar_filament` folder in your My Drive (`HANDOVER.md` §0.6).

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/solar_filament"
assert os.path.isdir(f"{DRIVE_DIR}/output/detectron2"), \
    "Team Drive folder not found: add the shortcut to Kimee's solar_filament folder (HANDOVER.md section 0.6)"
MY_DIR = f"{DRIVE_DIR}/Phase 6/Syaz"        # the ONLY place this notebook writes to
OUT_DIR = f"{MY_DIR}/output"                # checkpoints + raw predictions (persistent)
os.makedirs(OUT_DIR, exist_ok=True)
os.environ["SOLAR_OUTPUT_DIR"] = OUT_DIR
print("Writing to:", MY_DIR)

## 2. Get the latest code (pulls if the repo is already cloned)

In [ ]:
BRANCH = "Dev"
REPO = "github.com/amiruliman592/Kaggle-Competition-.git"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

if os.path.exists("/content/solar"):
    !git -C /content/solar pull -q origin {BRANCH}
else:
    !git clone -q --depth 1 -b {BRANCH} {url} /content/solar
%cd /content/solar
os.environ["SOLAR_ROOT"] = "/content/solar"
!git log --oneline -1

## 3. Install dependencies (Detectron2 build ~5–8 min)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
import torch, detectron2
print("torch", torch.__version__, "| detectron2", detectron2.__version__, "| GPU", torch.cuda.get_device_name(0))

## 4. Quick checks (~30 s) and helpers

In [ ]:
import shutil, subprocess, time

def sh(cmd):
    """Run one command (output shown live); stop the notebook if it fails."""
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd}")
    print(f"--- done in {(time.time() - t0) / 60:.1f} min")

def done(name):
    """True if this run's predictions are already saved (finished in an earlier session)."""
    return all(os.path.exists(f"{MY_DIR}/raw_preds_{name}_{s}.json.gz") for s in ("val", "test"))

def train_and_dump(name, flags):
    if done(name):
        print(f"{name}: already finished, skipped.")
        return
    sh(f"python run_pipeline.py --step train --name {name} --label-policy all {flags}")
    sh(f"python run_pipeline.py --step dump  --name {name}")
    for s in ("val", "test"):
        shutil.copy(f"{OUT_DIR}/raw_preds_{name}_{s}.json.gz", MY_DIR)
    print(f"{name}: predictions saved to {MY_DIR}")

sh("python -m pytest -q -p no:warnings tests/test_pipeline_flags.py tests/test_flip.py tests/test_official_metric.py")

## Step 0: probe (~2 min)
20 training steps of the exp006 recipe. Must pass, or the notebook stops (send Claude the error).

In [ ]:
sh("python run_pipeline.py --step train --name syaz_probe --label-policy all --max-iter 20 --no-resume")
shutil.rmtree(f"{OUT_DIR}/detectron2/syaz_probe", ignore_errors=True)
print("Probe passed.")

## S1: same recipe, other seed (`syaz_r50_seed2`, ~1.3 h)
Look for `Peak GPU memory` at the end of training and `[Dump] ... s/img` at the end.

In [ ]:
train_and_dump("syaz_r50_seed2", "--seed 7")

## S2: 3× longer training (`syaz_r50_3x`, ~4 h)
18,000 steps; the learning-rate drops move with it (at 60% and 85% of the steps). Seed 42 as in exp006, so the only
change is the length. If the session ends, re-run sections 1–4 and this cell: it resumes.

In [ ]:
train_and_dump("syaz_r50_3x", "--max-iter 18000")

## 5. Send results back
Download the prediction files (also in the team Drive `Phase 6/Syaz/`) into the repo folder
`Result Phase 6/Syaz/` on your laptop, then tell Claude: it scores them against exp006 (val PQ + Dice) on the laptop.
Also copy the `Peak GPU memory` and `[Dump] ... s/img` lines.
Then **disconnect the runtime** (Runtime → Disconnect and delete runtime) to save your Colab quota.

In [ ]:
from google.colab import files
for name in ("syaz_r50_seed2", "syaz_r50_3x"):
    for s in ("val", "test"):
        f = f"{MY_DIR}/raw_preds_{name}_{s}.json.gz"
        if os.path.exists(f):
            files.download(f)
        else:
            print("not found (run not finished?):", f)